[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/labs/L1/lab.ipynb)

# Lab 1: reglas para MiBici, la recta y tres varas

| | |
|---|---|
| **Parte 1** | Después de la primera pausa, unos 20 minutos: reglas para MiBici |
| **Parte 2** | Después de la pausa larga, unos 20 minutos: la recta con scikit-learn |
| **Parte 3** | Al final de la sesión, unos 50 minutos: ¿qué tan buena es la recta? Tres varas |
| **Qué se califica** | Las celdas que dicen **Entregable** y tus **dos interpretaciones** (la mezcla y las tres varas) |
| **Ritmo** | Lo hacemos juntos, al mismo ritmo: yo corro, ustedes corren, y en cada *Ahora tú* todos tenemos los mismos minutos. Lo que falte, en casa |
| **Entrega** | En Moodle, domingo 8 de noviembre a las 23:59. Se acepta hasta el miércoles 11. |

**Antes de empezar**
1. En *Archivo*, *Guardar una copia en Drive*, y trabaja en esa copia. Si no, al cerrar pierdes tus cambios.
2. Corre las dos celdas de abajo hasta que aparezca `✅ entorno listo`.
3. Si algo falla: *Entorno de ejecución*, *Reiniciar y ejecutar todo*.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES

du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

## Parte 1: reglas para MiBici
Cada fila es **una estación en una hora**; la respuesta $y$ es `departures`, cuántas bicis salieron. `train` (octubre a junio) son los ejemplos que vemos; `julio`, los datos nuevos para comparar. Agosto y septiembre no se tocan.

### En vivo: cargar los datos

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
train = df[df.particion == "train"]   # octubre a junio
julio = df[df.particion == "val"]     # julio
print(df.shape)
print("meses:", sorted(df.month.unique().tolist()))
print("de", df.hour.min(), "a", df.hour.max())

# cuántas filas tiene cada mes y a qué partición va
conteo = pd.crosstab(df.hour.dt.to_period("M"), df.particion)
conteo.plot(kind="bar", figsize=(10, 3.5), width=0.8)
plt.title("Registros por mes y partición")
plt.xlabel("mes")
plt.ylabel("filas")
plt.legend(title="partición")
plt.tight_layout()
plt.show()

df[["station_id", "hour", "departures", "departures_lag_24h", "departures_lag_168h"]].head()

**¿Qué esperamos ver?** Doce meses, de octubre de 2025 a septiembre de 2026. Octubre a junio son `train`, julio es `val` (lo llamamos `julio`) y agosto y septiembre son `test`, que no se tocan hasta el final del curso.

**¿Qué son `departures_lag_24h` y `departures_lag_168h`?** Las salidas de esa misma estación, a esa misma hora, ayer y hace una semana. *Lag* quiere decir rezago. Los rezagos ya están calculados: 24 h es ayer a la misma hora; 168 h, hace una semana.

`departures_mean_168h` es el promedio de los 7 días que terminan 24 horas antes.

### En vivo: qué es `train.departures`
`train.departures` es la columna $y$: las bicis que salieron de cada estación en cada hora, de octubre a junio. Es lo que queremos predecir. Veamos unas filas, cómo se reparten los valores y una semana de la estación con más salidas.

In [ ]:
display(train[["station_id", "hour", "departures"]].head())

est = train.groupby("station_id").departures.mean().idxmax()   # la estación con más salidas
semana = train[(train.station_id == est) & (train.hour >= "2026-03-02") & (train.hour < "2026-03-09")]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.hist(train.departures, bins=range(0, 31), color=COLORES["w"])
ax1.set_xlabel("salidas en una hora")
ax1.set_ylabel("filas")
ax1.set_title("train.departures, todas las filas")
ax2.plot(semana.hour, semana.departures, color=COLORES["tinta"], linewidth=1.5)
ax2.set_xlabel("hora (2 al 8 de marzo)")
ax2.set_ylabel("salidas")
ax2.set_title(f"Estación {est}, una semana")
ax2.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

**¿Qué esperamos ver?** Casi un tercio de las filas vale 0 (de madrugada no sale nadie) y la mayoría está entre 0 y 10. En la semana, un pico en la mañana y otro en la tarde cada día hábil, y un patrón distinto el sábado y el domingo.

### En vivo: aplicar una regla
La regla nos la dan: "saldrán las mismas que hace una semana". Aplicarla es **deducir** la predicción. Antes de correr: ¿por cuántas bicis crees que se equivoca?

In [ ]:
y_hat = train.departures_lag_168h
error = y_hat - train.departures      # predicción menos respuesta
print(f"MAE en train: {np.mean(np.abs(error)):.2f}")

# la misma semana de arriba: lo que salió y lo que dijo la regla
plt.figure(figsize=(12, 3.5))
plt.plot(semana.hour, semana.departures, color=COLORES["tinta"], linewidth=1.5, label="real (departures)")
plt.plot(semana.hour, semana.departures_lag_168h, color=COLORES["yhat"], linewidth=1.5, linestyle="--",
         label="regla: hace una semana")
plt.ylabel("salidas")
plt.title(f"Estación {est}: la regla contra lo que pasó")
plt.legend()
plt.tight_layout()
plt.show()

**¿Qué esperamos ver?** La regla copia la forma de la semana anterior: acierta en cuándo hay picos, pero falla en qué tan altos son. La distancia vertical entre las dos líneas es el error de cada hora.

**MAE, el error absoluto medio:** $\text{MAE} = \frac{1}{m}\sum_{i=1}^{m} \big|\hat y^{(i)} - y^{(i)}\big|$

El promedio de los errores sin signo, para que no se cancelen. Está en bicis: un MAE de 2.3 es fallar por 2.3 bicis por hora, en promedio. Con él comparamos todo lo que sigue, de las reglas a la recta.

### En vivo: escoger del menú
Ahora los datos escogen la regla: eso es **inducir**. Escoge cada regla del menú y anota su MAE en train. Antes de correr: ¿cuál gana?

In [ ]:
menu = {
    "igual que hace una semana": "departures_lag_168h",
    "igual que ayer": "departures_lag_24h",
    "promedio de la última semana": "departures_mean_168h",
}
mae_train = {nombre: np.mean(np.abs(train[col] - train.departures)) for nombre, col in menu.items()}


def ver_regla(regla):
    print(f"{regla} ({menu[regla]}): MAE en train {mae_train[regla]:.2f}")

In [ ]:
try:
    import ipywidgets as widgets
    widgets.interact(ver_regla, regla=widgets.Dropdown(options=list(menu), description="Regla"))
except ImportError:
    print("No hay ipywidgets: usa la celda de abajo.")

In [ ]:
# Si el widget no aparece: cambia la regla y vuelve a correr esta celda
ver_regla("igual que hace una semana")

### En vivo: ¿cuál gana?
Las tres juntas. Gana la de menor MAE en train; julio solo lo vemos para comprobar.

In [ ]:
ganadora = min(mae_train, key=mae_train.get)          # la de menor MAE en train
col_ganadora = menu[ganadora]
mae_julio_ganadora = np.mean(np.abs(julio[col_ganadora] - julio.departures))

plt.figure(figsize=(7, 3.5))
barras = plt.bar(list(mae_train), list(mae_train.values()),
                 color=[COLORES["yhat"] if n == ganadora else "gray" for n in mae_train])
plt.bar_label(barras, fmt="%.2f")
plt.ylabel("MAE en train (bicis por hora)")
plt.title("El menú de reglas (menor es mejor)")
plt.tight_layout()
plt.show()
print(f"Gana '{ganadora}'. En julio: {mae_julio_ganadora:.2f}")

**¿Qué esperamos ver?** Gana "igual que hace una semana" con 2.20 en train y 2.30 en julio. El promedio de la semana queda lejos (3.26): promedia todas las horas, madrugada y hora pico, así que da casi el mismo número a cualquier hora.

### Ahora tú: mezclar dos reglas
Una regla nueva a partir de dos. Si $w$ es el peso de la regla A (el *Peso de A* del widget):

$$\hat y = w \times (\text{regla A}) + (1 - w) \times (\text{regla B})$$

- Con $w = 1$, la predicción es la regla A sola.
- Con $w = 0$, es la regla B sola.
- Con $w = 0.5$, es el promedio de las dos.

**¿Mezclar dos reglas le gana a la mejor regla sola? ¿Con qué peso?**

*Tip: escoge el peso viendo la curva de train; el MAE de julio que aparece abajo es para comprobar.*

**Nota:** esto no es regresión polinomial; nada se eleva al cuadrado. Es una **combinación lineal** de dos columnas, un promedio ponderado. La regresión lineal hace lo mismo con más libertad: le da un peso a cada columna, suma una $b$ y escoge los pesos sola. Eso es la parte 2.

In [ ]:
import functools

reglas = {"ayer (departures_lag_24h)": "departures_lag_24h",
          "hace una semana (departures_lag_168h)": "departures_lag_168h",
          "promedio de la semana (departures_mean_168h)": "departures_mean_168h"}
pesos = np.round(np.arange(0, 1.01, 0.1), 1)


def mae_mezcla(datos, a, b, peso):
    y_hat = peso * datos[a] + (1 - peso) * datos[b]
    return np.mean(np.abs(y_hat - datos.departures))


@functools.lru_cache
def curvas(a, b):
    return ([mae_mezcla(train, a, b, p) for p in pesos], [mae_mezcla(julio, a, b, p) for p in pesos])


def ver_mezcla(regla_a, regla_b, peso):
    a, b = reglas[regla_a], reglas[regla_b]
    if a == b:
        print("Escoge dos reglas distintas.")
        return
    c_train, c_julio = curvas(a, b)
    k = int(round(peso * 10))
    plt.figure(figsize=(7, 4))
    plt.plot(pesos, c_train, marker="o", label="train")
    plt.axhline(mae_train[ganadora], linestyle="--", color="gray", label=f"ganadora sola, en train: {mae_train[ganadora]:.2f}")
    plt.scatter([peso], [c_train[k]], s=120, color="black", zorder=3)
    corto_a, corto_b = regla_a.split(" (")[0], regla_b.split(" (")[0]
    plt.xlabel(f"peso de A: 0 = solo '{corto_b}', 1 = solo '{corto_a}'")
    plt.ylabel("MAE en train (bicis por hora)")
    plt.legend()
    plt.tight_layout()
    plt.show()
    print(f"Peso {peso:.1f}: MAE en train {c_train[k]:.2f}")
    print(f"Para comprobar, en julio: {c_julio[k]:.2f} (la ganadora sola en julio: {mae_julio_ganadora:.2f})")


try:
    import ipywidgets as widgets
    widgets.interact(ver_mezcla,
                     regla_a=widgets.Dropdown(options=list(reglas), value="ayer (departures_lag_24h)", description="Regla A"),
                     regla_b=widgets.Dropdown(options=list(reglas), value="hace una semana (departures_lag_168h)", description="Regla B"),
                     peso=widgets.FloatSlider(value=0.5, min=0, max=1, step=0.1, description="Peso de A"))
except ImportError:
    print("No hay ipywidgets: usa la celda de abajo.")

In [ ]:
# Si el widget no aparece: cambia los valores y vuelve a correr esta celda
ver_mezcla("ayer (departures_lag_24h)", "hace una semana (departures_lag_168h)", 0.5)

**¿Qué esperamos ver?** Una curva del MAE en train según el peso. Buscas su punto más bajo: ahí la mezcla se equivoca menos.

**¿Y la línea gris?** Es el MAE en train de la regla ganadora sola. Si el fondo de la curva queda debajo de la línea gris, mezclar le gana a la mejor regla sola.

### Entregable 1: tu mezcla
Escribe lo que escogiste con el widget: las dos reglas, el peso, el MAE en julio que te dio y si le gana a la ganadora sola. El evaluador revisa que tu peso sea el del fondo de la curva de **train** para ese par, con un margen de 0.02 de MAE. Las reglas van entre comillas, con el nombre que muestra el widget o con el de la columna: `"ayer (departures_lag_24h)"`, `"ayer"` y `"departures_lag_24h"` valen lo mismo.

Escribe el valor tal cual (por ejemplo `0.4`, no `4/10` ni una variable): el evaluador solo lee valores escritos.

In [ ]:
# Entregable 1: tu mezcla. Escribe lo que escogiste con el widget.
# Escribe el valor tal cual (por ejemplo 0.4, no 4/10 ni una variable): el evaluador solo lee valores escritos.
REGLA_A = ""        # como en el widget o con la columna: "ayer (departures_lag_24h)", "ayer" o "departures_lag_24h"
REGLA_B = ""
PESO = None         # de 0 a 1, en pasos de 0.1: cuánto pesa REGLA_A
MAE_JULIO = None    # el MAE en julio que te dio el widget con esa mezcla
LE_GANA = None      # True o False: ¿le gana a la regla ganadora sola en julio?

**¿Por qué escogiste esa mezcla y ese peso?** En 2 o 3 líneas, con tus números.

*Tip: ¿lo escogiste viendo train o viendo julio? ¿Por qué importa?*

_Escribe aquí._

### Una regla por estación y hora
El promedio de salidas de cada estación, a cada hora, entre semana o en fin de semana, calculado con `train`. Ya es una regla **inducida** con 254 mil ejemplos. **Córrela aunque vayas tarde:** la usamos en la parte 3.

**¿Por qué `is_weekend`?** Entre semana las bicis siguen la rutina de trabajo y escuela; en fin de semana el uso es de paseo y cambia de hora. Con `is_weekend` la tabla guarda un promedio para cada caso en lugar de revolverlos.

In [ ]:
claves = ["station_id", "hour_of_day", "is_weekend"]
prom = train.groupby(claves).departures.mean().rename("y_hat")

# Veamos cómo luce por dentro: el promedio a las 8 de la mañana, por estación y tipo de día
display(prom.xs(8, level="hour_of_day").unstack("is_weekend")
        .rename(columns={0: "entre semana", 1: "fin de semana"}).round(2).head(6))

y_tabla = julio.join(prom, on=claves).y_hat
print("filas de julio sin promedio:", y_tabla.isna().sum())
print(f"MAE en julio: {np.mean(np.abs(y_tabla - julio.departures)):.2f}")

**La tabla como fórmula.** Para cada combinación de estación, hora y tipo de día, la predicción es el promedio de las $y$ de train que caen en ese grupo:

$$\hat y(\text{estación}, \text{hora}, \text{tipo de día}) = \frac{1}{N}\sum_{i\,\in\,\text{grupo}} y^{(i)}$$

$N$ es cuántas veces aparece esa combinación en `train`. Es un promedio condicional. Ahora, contra la ganadora del menú:

In [ ]:
maes = {"igual que hace una semana": mae_julio_ganadora,
        "tabla por estación y hora": np.mean(np.abs(y_tabla - julio.departures))}
plt.figure(figsize=(6, 3.5))
barras = plt.bar(list(maes), list(maes.values()), color=["gray", COLORES["yhat"]])
plt.bar_label(barras, fmt="%.2f", fontweight="bold")
plt.ylabel("MAE en julio (bicis por hora)")
plt.title("Dos reglas en julio (menor es mejor)")
plt.tight_layout()
plt.show()

**¿Qué esperamos ver?**
1. En la tabla, la misma estación a la misma hora tiene un promedio distinto entre semana y en fin de semana.
2. Cero filas de julio sin promedio: cada combinación de julio ya apareció en train.
3. Un MAE en julio de 1.95 contra 2.30 de la ganadora. Es una regla difícil de superar; la vamos a usar de referencia.

### Tus preguntas (parte 1)
- **El error:** si una regla tuviera MAE de 0.0, ¿qué querría decir en la vida real? ¿Crees que se puede con algo tan humano como MiBici?
- **La mezcla:** ¿cómo puede una mezcla de dos reglas equivocarse menos que la mejor de las dos? Pista: piensa en una regla que se pasa y otra que se queda corta.
- **Los datos:** ¿por qué comprobamos en julio en lugar de quedarnos con lo que dijo train?

Anota una duda para la siguiente sesión. No se califica.

_Escribe aquí._

## Parte 2: la recta de MiBici con scikit-learn
Después de la pausa larga. Primero la usamos; en la parte 3 medimos qué tan buena quedó.

### En vivo: la recta en tres líneas
`fit` entrena, `coef_` e `intercept_` son $w$ y $b$, `predict` calcula $w\,x + b$. `X` va con doble corchete porque scikit-learn espera una tabla, aunque tenga una sola columna.

**¿Cómo se ve?** En el eje horizontal, las salidas de hace una semana; en el vertical, las de hoy. La recta pasa entre esos puntos.
- $w$ es la **inclinación**: por cada bici más hace una semana, cuántas más esperamos hoy.
- $b$ es donde la recta corta el eje vertical: si hace una semana salieron 0, cuántas esperamos hoy.

In [ ]:
from sklearn.linear_model import LinearRegression

X = train[["departures_lag_168h"]]
modelo = LinearRegression().fit(X, train.departures)
print(f"w = {modelo.coef_[0]:.2f}, b = {modelo.intercept_:.2f}")

y_recta = modelo.predict(julio[["departures_lag_168h"]])
mae_recta = np.mean(np.abs(y_recta - julio.departures))
print(f"MAE en julio: {mae_recta:.2f}")

# 3,000 filas de train al azar, la recta y la regla (hoy = hace una semana)
muestra = train.sample(3000, random_state=0)
xs = np.array([0, 40])
plt.figure(figsize=(6, 5))
plt.scatter(muestra.departures_lag_168h, muestra.departures, s=10, alpha=0.3, color=COLORES["w"], edgecolors="none")
plt.plot(xs, modelo.coef_[0] * xs + modelo.intercept_, color=COLORES["yhat"],
         label=f"recta: {modelo.coef_[0]:.2f} x + {modelo.intercept_:.2f}")
plt.plot(xs, xs, color="gray", linestyle="--", label="regla: hoy = hace una semana")
plt.xlabel("salidas hace una semana (x)")
plt.ylabel("salidas hoy (y)")
plt.legend()
plt.tight_layout()
plt.show()

**¿Qué esperamos ver?** $w = 0.70$ y $b = 1.18$: por cada bici de hace una semana, la recta espera 0.7 hoy, más 1.18 de base. Es más plana que la regla (la línea gris, $w = 1$ y $b = 0$): sube los valores chicos y baja los grandes, hacia el promedio. En julio su MAE es 2.35, un poco peor que la regla (2.30); en la parte 3 vemos que con otra vara la historia cambia.

| En las slides | En scikit-learn |
|---|---|
| buscar el fondo del costo $J$ | `modelo.fit(X, y)` |
| $w$, $b$ | `modelo.coef_`, `modelo.intercept_` |
| $\hat y = w\,x + b$ | `modelo.predict(X)` |

### Ahora tú: el reto de las columnas
`prueba_columnas` entrena una recta con las columnas que le des (con `train`) y la mide en julio. La hora, el día y la estación las trata como categorías: la hora se vuelve 24 columnas de 0 y 1, y la recta aprende un ajuste para cada hora. Julio aquí sirve para comparar propuestas; agosto y septiembre no se tocan.

**¿Qué columnas le dan a la recta el MAE más bajo en julio?**

*Tip: antes de escoger, escribe qué MAE esperas. Prueba al menos tres listas; más columnas no siempre es mejor.*

In [ ]:
CATEGORIAS = ["hour_of_day", "day_of_week", "station_id"]
COLUMNAS = ["departures_lag_24h", "departures_lag_168h", "departures_mean_24h", "departures_mean_168h",
            "hour_of_day", "day_of_week", "is_weekend", "station_id"]
intentos = {}


def tabla_x(datos, columnas):
    partes = []
    for col in columnas:
        if col in CATEGORIAS:
            cats = sorted(df[col].unique())
            partes.append(pd.get_dummies(pd.Categorical(datos[col], categories=cats), prefix=col)
                          .astype(float).set_index(datos.index))
        else:
            partes.append(datos[[col]].astype(float))
    return pd.concat(partes, axis=1)


def prueba_columnas(columnas):
    columnas = list(columnas)
    if not columnas:
        print("Escoge al menos una columna.")
        return
    m = LinearRegression().fit(tabla_x(train, columnas), train.departures)
    mae_julio = np.mean(np.abs(m.predict(tabla_x(julio, columnas)) - julio.departures))
    intentos[", ".join(columnas)] = round(mae_julio, 2)
    print(f"MAE en julio: {mae_julio:.2f}   con {columnas}")

Escoge columnas en la lista; con Ctrl (Cmd en Mac) escoges varias. Cada combinación se entrena al momento y queda en tu historial.

In [ ]:
try:
    import ipywidgets as widgets
    widgets.interact(prueba_columnas,
                     columnas=widgets.SelectMultiple(options=COLUMNAS, value=["departures_lag_168h", "hour_of_day"],
                                                     rows=len(COLUMNAS), description="Columnas"))
except ImportError:
    print("No hay ipywidgets: usa la celda de abajo.")

In [ ]:
# Si el widget no aparece, escribe aquí cada lista que pruebes y vuelve a correr esta celda.
# También sirve para que tus listas no se pierdan al "Reiniciar y ejecutar todo".
mis_listas = [
    ["departures_lag_168h", "hour_of_day"],
]
for columnas in mis_listas:
    prueba_columnas(columnas)

In [ ]:
# Tu historial de intentos, del mejor al peor
if intentos:
    display(pd.Series(intentos, name="MAE en julio").sort_values().to_frame())
else:
    print("Todavía no pruebas ninguna lista.")

**¿Qué esperamos ver?** Con solo `departures_lag_168h` la recta da 2.35. Con la hora baja a 2.09 y con el rezago de ayer a 2.13; el día, el fin de semana o la estación casi no lo mueven. Con todas juntas no sale el MAE más bajo. La primera fila del historial es tu mejor lista: la usamos en la parte 3.

### Tus preguntas (parte 2)
- **Los coeficientes:** con solo `departures_lag_168h` la recta dio $w = 0.70$ y $b = 1.18$. ¿Qué quiere decir cada número para MiBici?
- **Las categorías:** con `hour_of_day` la recta recibe 24 columnas de 0 y 1 en lugar de un número del 0 al 23. ¿Qué saldría mal si usara la hora como número?
- **Más columnas:** ¿por qué meterle todas las columnas no siempre da el MAE más bajo en julio?

Anota una duda para la siguiente sesión. No se califica.

_Escribe aquí._

## Parte 3: ¿qué tan buena es la recta? Tres varas
Al final de la sesión. Medimos los mismos modelos con tres varas distintas y vemos si el ganador cambia.

| Vara | Fórmula | En palabras | En scikit-learn |
|---|---|---|---|
| MAE | $\frac{1}{m}\sum \lvert\hat y - y\rvert$ | el error típico, en bicis | `mean_absolute_error(y, y_hat)` |
| RMSE | $\sqrt{\frac{1}{m}\sum (\hat y - y)^2}$ | igual, pero los errores grandes pesan más | `root_mean_squared_error(y, y_hat)` |
| $R^2$ | $1 - \frac{\sum (\hat y - y)^2}{\sum (\bar y - y)^2}$ | qué fracción del error de "decir siempre el promedio" le quitaste | `r2_score(y, y_hat)` |

### En vivo: las tres varas con las tortillas
La recta 25x contra decir siempre el promedio, 50 pesos. **¿Qué dice cada vara? ¿Cuáles se leen en pesos y cuál no?**

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

x_t = np.array([1, 2, 3])        # kilos
y_t = np.array([25, 51, 74])     # pesos
for nombre, y_hat in [("recta 25x", 25 * x_t), ("siempre el promedio", np.full(3, y_t.mean()))]:
    print(f"{nombre:20s} MAE = {mean_absolute_error(y_t, y_hat):5.2f}   RMSE = {root_mean_squared_error(y_t, y_hat):5.2f}"
          f"   R² = {r2_score(y_t, y_hat):6.3f}")

**¿Qué esperamos ver?** La recta: MAE 0.67 y RMSE 0.82, en pesos, y R² 0.998. El promedio: MAE 16.67 y RMSE 20.02 pesos, y R² 0, porque se compara contra sí mismo. MAE y RMSE se leen en pesos; R² no tiene unidades.

### En vivo: la métrica decide quién gana
MiBici en julio: la regla "hace una semana" contra la recta con esa misma columna. **Con MAE gana la regla. ¿Y con RMSE?**

In [ ]:
def tres_varas(nombre, y_hat):
    y = julio.departures
    print(f"{nombre:28s} MAE = {mean_absolute_error(y, y_hat):.2f}   RMSE = {root_mean_squared_error(y, y_hat):.2f}"
          f"   R² = {r2_score(y, y_hat) + 0.0:.2f}")


tres_varas("siempre el promedio de train", np.full(len(julio), train.departures.mean()))
tres_varas("regla: hace una semana", julio.departures_lag_168h)
tres_varas("recta: hace una semana", y_recta)

**¿Qué esperamos ver?** Con MAE gana la regla (2.30 contra 2.35); con RMSE gana la recta (3.84 contra 4.16) y su R² es más alto (0.50 contra 0.41). Decir siempre el promedio de train da R² de casi 0 en julio.

### Ahora tú: tu mejor recta con las tres varas
La celda toma sola tu mejor lista del reto (parte 2) y la mide con las tres varas.

**¿Tu recta le gana a la regla? ¿Le gana con las tres?**

*Tip: si el RMSE baja más que el MAE, tu recta tiene menos errores grandes.*

In [ ]:
# tu mejor lista del reto; si no hay intentos, la lista con la que empieza el reto
columnas = min(intentos, key=intentos.get).split(", ") if intentos else ["departures_lag_168h", "hour_of_day"]
print("tu mejor lista:", columnas)
m = LinearRegression().fit(tabla_x(train, columnas), train.departures)
tres_varas("tu recta", m.predict(tabla_x(julio, columnas)))

**¿Qué esperamos ver?** Si sumaste la hora o el rezago de ayer, tu recta le gana a la regla con las tres varas. Compara cuánto baja el MAE y cuánto el RMSE.

### Todo junto, en julio
Lo de hoy en una tabla, para tu interpretación.

In [ ]:
resumen = pd.Series({
    "igual que hace una semana": mae_julio_ganadora,
    "tabla por estación y hora": np.mean(np.abs(y_tabla - julio.departures)),
    "recta, una columna": mae_recta,
    **({f"tu mezcla ({REGLA_A}, {REGLA_B}, peso {PESO})": MAE_JULIO} if MAE_JULIO else {}),
    **({f"recta, tu mejor intento ({min(intentos, key=intentos.get)})": min(intentos.values())} if intentos else {}),
}, name="MAE en julio").round(2)
resumen.sort_values()

### Entregable 2
Dos de las tres varas, en Python puro: listas, `for`, `sum` y `math.sqrt`, sin NumPy.

**¿Qué esperamos ver?** La celda de prueba usa los tickets de las tortillas (kilos y pesos). Si tu código está bien:
- **RMSE de 0.82:** cobrar 25 pesos por kilo se equivoca por unos 82 centavos, con los errores grandes pesando más (el MAE es 0.67).
- **R² de 0.998:** usar los kilos le quita el 99.8 % del error que tendría cobrarle a todos el promedio.

In [ ]:
import math


def rmse(y_hat, y):
    """Raíz del error cuadrático medio: √( promedio de (ŷ − y)² ). Recibe dos listas del mismo largo."""
    # TODO: cada error al cuadrado, el promedio y al final math.sqrt
    raise NotImplementedError("Completa rmse")


def r2(y_hat, y):
    """R² = 1 − Σ(ŷ − y)² / Σ(ȳ − y)², con ȳ el promedio de y. Recibe dos listas del mismo largo.
    Si todas las y son iguales: 1.0 si el modelo acierta todas, 0.0 si no (como scikit-learn)."""
    # TODO: el error al cuadrado del modelo, el de decir siempre el promedio, y 1 − uno entre otro
    raise NotImplementedError("Completa r2")

In [ ]:
try:
    print("rmse de 25x con los tickets (0.82):", round(rmse([25, 50, 75], [25, 51, 74]), 2))
    print("r2 de 25x con los tickets (0.998):", round(r2([25, 50, 75], [25, 51, 74]), 3))

except NotImplementedError as e:
    print('Todavía falta:', e)

### Tu interpretación
En 3 a 5 líneas y con tus números: compara la regla "hace una semana" con la **recta de una columna** (la de la parte 2). Con MAE gana la regla y con RMSE gana la recta. ¿Por qué no coinciden? ¿Qué vara usarías para MiBici y por qué?

_Escribe aquí._

### Tus preguntas (parte 3)
- **Errores grandes:** el RMSE nunca es menor que el MAE. Si en un modelo el RMSE es mucho más grande que el MAE y en otro casi igual, ¿qué te dice de sus errores?
- **R² negativo:** decir siempre el promedio da R² de 0. Si un modelo sacara R² negativo en julio, ¿qué querría decir de sus predicciones?
- **MiBici:** si fueras quien reparte las bicis, ¿cuándo te importaría más el RMSE (evitar errores enormes) y cuándo el MAE (bicis de error por hora, en promedio)?

Anota una duda para la siguiente sesión. No se califica.

_Escribe aquí._

**Para entregar:** *Entorno de ejecución*, *Reiniciar y ejecutar todo*; luego *Archivo*, *Descargar .ipynb*, y súbelo a Moodle. No cambies el nombre de las funciones ni de las variables del Entregable 1.